# 10주차 실습 — MLP 제대로 학습시키기: 손글씨 숫자 60,000장 ✍️

9주차에는 데이터 150점·파라미터 49개짜리 회귀를 PyTorch로 돌렸습니다. 오늘은 **이미지 60,000장·파라미터 10만 개·정답이 10종류인 분류** 문제를 같은 4조각(모델·손실·옵티마이저·루프)으로 학습시킵니다. 새로 등장하는 것은 네 가지뿐입니다 — ① 출력 10개와 `CrossEntropyLoss` ② 미니배치 `DataLoader` ③ 훈련/검증 **손실 곡선 두 개**로 과적합을 보는 법 ④ 과적합 대책(조기 종료·드롭아웃)과 모델 저장.

**학습 목표**
1. 이미지가 신경망에 들어가는 모양(28×28 → 784개 입력)과 분류 출력(10개 점수 → softmax → 확률)을 읽는다
2. 훈련/검증 손실 곡선이 **벌어지는 지점**을 과적합으로 읽고, 조기 종료·드롭아웃이 곡선을 어떻게 바꾸는지 본다
3. 학습한 모델을 **파일로 저장하고 다시 불러와** 같은 정확도가 나오는지 확인한다

**데이터**: MNIST(손글씨 숫자 0~9, 28×28 흑백, 훈련 60,000 / 테스트 10,000) — `torchvision`이 자동으로 내려받습니다. 업로드할 파일 없음.
**기계공학에서는**: 계기판·명판·각인 번호의 숫자 판독(OCR), 부품 표면 결함 사진 분류(11주차 CNN) 등 "픽셀 → 범주" 문제의 가장 작은 교과서 예제입니다.
**런타임**: CPU로 전체 10분 이내. (런타임 > 런타임 유형 변경 > T4 GPU 를 켜면 더 빠릅니다 — 11주차부터 필수)
**선행**: 보충 `week10_lab_python.ipynb`(키워드 인자·with 문)

> 막히면: ① 에러 메시지를 복사해 AI에게 ② 런타임 > 모두 실행

## 0. 환경 준비 — MNIST 내려받기

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, copy, time
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"        # GPU가 있으면 쓰고 없으면 CPU (11주차 예고)
print("PyTorch", torch.__version__, "| device:", device)

transform = transforms.ToTensor()                             # 이미지(0~255 정수) → 텐서(0.0~1.0 실수)로 바꾸는 부품
train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)   # 60,000장
test_ds    = datasets.MNIST(root="./data", train=False, download=True, transform=transform)   # 10,000장 — 🔒 마지막에 한 번만
print("훈련 전체:", len(train_full), "장 / 테스트:", len(test_ds), "장")

## 개념 요약 (1차시 복습)

- **분류**는 출력이 **클래스 수만큼**(10개)이고, 각 출력은 "그 숫자일 점수(logit)". **softmax**가 점수 10개를 합이 1인 확률로 바꾸고, 가장 큰 칸이 예측. 3주차 로지스틱 회귀(시그모이드, 2클래스)의 10클래스판
- 손실은 **`nn.CrossEntropyLoss`** = softmax + 로그 손실을 한 부품에. 정답은 **정수 라벨 그대로**(0~9) 넣는다 — 원-핫으로 바꾸지 않는다
- 데이터가 많으면 한 번에 다 넣지 않고 **미니배치**(예: 128장)씩 넣어 한 걸음씩 — 전체를 한 바퀴 돈 것이 **에폭**. `DataLoader`가 섞어서 나눠 준다
- **훈련/검증/테스트**(6주차): 검증 세트로 에폭 수·하이퍼파라미터를 고르고 테스트는 마지막 한 번. **훈련 손실은 내려가는데 검증 손실이 올라가기 시작하면 과적합** — 그 지점에서 멈추는 것이 **조기 종료**, 학습 중 뉴런을 무작위로 꺼서 외우기를 방해하는 것이 **드롭아웃**
- 학습한 가중치는 `state_dict()`를 **파일로 저장**해 현장(다른 컴퓨터)에서 불러 쓴다

## 🐍 파이썬 5분 문법 — 키워드 인자 읽기 · `with` 문 (읽기 훈련)

| 모양 | 읽는 법 | 오늘 어디에 |
|---|---|---|
| `DataLoader(ds, batch_size=128, shuffle=True)` | 첫 인자만 위치로, 나머지는 **이름=값**(키워드 인자). 안 쓴 인자(`num_workers` 등)는 **기본값** | 3단계 |
| `torch.optim.Adam(model.parameters(), lr=1e-3)` | `1e-3` = 0.001 (과학 표기). `lr`을 안 쓰면 기본값 0.001 | 5단계 |
| `with torch.no_grad():` … | "이 블록 **안에서만** 기울기 추적을 끈다" — 블록이 끝나면 자동으로 원래대로 | 5단계 평가 |
| `with open("history.json", "w") as f:` … | "파일을 열어 `f`라 부르고, 블록이 끝나면 **자동으로 닫는다**" — 닫는 것을 잊을 수 없게 하는 문법 | 10단계 저장 |

`with`는 "들어갈 때 준비, 나올 때 뒷정리"를 자동으로 해 주는 문법입니다. 파일(열기/닫기), 기울기 추적(끄기/켜기) 둘 다 같은 모양입니다.

In [ ]:
# 키워드 인자: 이름을 쓰면 순서를 몰라도 읽힌다. 기본값은 help()로 확인
import inspect
sig = inspect.signature(DataLoader.__init__)
print("DataLoader의 인자와 기본값 (앞 6개):", [f"{k}={v.default}" if v.default is not inspect._empty else k for k, v in list(sig.parameters.items())[1:7]])

# with 문: 블록 안에서만 유효하고 나오면 자동 정리
with open("memo.txt", "w") as f:          # 🐍 열고
    f.write("10주차 MNIST\n")                 #    쓰고
print("파일이 닫혔나:", f.closed)            #    블록을 나오자마자 자동으로 닫힘 → True

w = torch.tensor(1.0, requires_grad=True)
with torch.no_grad():                        # 🐍 같은 모양 — 블록 안에서만 추적 끔
    print("블록 안 (w*2).requires_grad:", (w * 2).requires_grad)
print("블록 밖 (w*2).requires_grad:", (w * 2).requires_grad)

✏️ **한 줄 연습**: `DataLoader(test_ds, batch_size=1000)`에서 `shuffle`은 몇으로 들어갈까요? 위 출력의 기본값을 읽어 보세요. (테스트 데이터는 섞을 필요가 없으니 기본값이 맞습니다)

# Part A — 데이터를 눈으로 보기 👀

## 1단계 — 이미지 16장과 텐서 모양

> **프롬프트 예시**
> "train_full에서 앞 16장을 4×4 격자로 그리고 각 제목에 라벨을 표시해줘. 첫 장의 텐서 shape, dtype, 최솟값·최댓값도 출력해줘.\"

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.ravel()):
    img, label = train_full[i]                            # 📦 (이미지 텐서, 정수 라벨) 튜플 — 🐍 5주차 언패킹
    ax.imshow(img.squeeze(), cmap="gray"); ax.set_title(f"label = {label}", fontsize=9); ax.axis("off")
plt.suptitle("MNIST — 16 training images"); plt.tight_layout(); plt.show()

img, label = train_full[0]
print("한 장의 텐서 shape:", tuple(img.shape), "= (채널 1, 세로 28, 가로 28) | dtype:", img.dtype, "| 값 범위:", img.min().item(), "~", img.max().item())
print("\n가운데 8×8 픽셀 값 (0 = 검정, 1 = 흰색):"); print(np.round(img[0, 10:18, 10:18].numpy(), 1))

💬 **관찰 질문**
1. 이미지 한 장은 **28×28 = 784개의 숫자**입니다. 5주차 기계 한 대가 센서 8개의 숫자였듯, 이미지 한 장은 "센서 784개"의 측정값이라고 보면 지금까지의 모든 도구가 그대로 적용됩니다
2. 픽셀 값은 0~1로 이미 **같은 단위·같은 범위** — 그래서 오늘은 StandardScaler 없이 바로 넣습니다(9주차 표준화 콜백: "처음부터 같은 단위면 생략")

## 2단계 — 784개로 펴기(flatten)와 클래스 분포

MLP(`nn.Linear`)는 (샘플 수 × 특성 수)의 **표**를 받습니다. 28×28 격자를 한 줄 784개로 펴는 부품이 `nn.Flatten()`입니다. 격자 위치 정보는 사라지지만 — 그래서 11주차 CNN이 필요해집니다 — 오늘은 이 방식으로도 97%를 넘깁니다.

> **프롬프트 예시**
> "nn.Flatten()을 한 장(1,28,28)과 배치(16,1,28,28)에 적용한 뒤의 shape를 출력하고, train_full의 라벨 분포(0~9 개수)를 막대그래프로 그려줘.\"

In [ ]:
flat = nn.Flatten()                                        # 📦 (N, 1, 28, 28) → (N, 784). 첫 차원(샘플)은 그대로 두고 나머지를 한 줄로
print("한 장:", tuple(img.unsqueeze(0).shape), "→", tuple(flat(img.unsqueeze(0)).shape))
batch16 = torch.stack([train_full[i][0] for i in range(16)])   # 🐍 5주차 컴프리헨션으로 16장 모으기
print("16장:", tuple(batch16.shape), "→", tuple(flat(batch16).shape), " ← nn.Linear(784, …)가 받을 모양")

labels_all = train_full.targets.numpy()                     # 📦 길이 60000 정수 배열 (텐서 → NumPy)
counts = np.bincount(labels_all)
plt.figure(figsize=(7, 3)); plt.bar(range(10), counts); plt.xticks(range(10)); plt.xlabel("digit"); plt.ylabel("count"); plt.title("Class balance — roughly even"); plt.show()
print("클래스별 개수:", dict(enumerate(counts.tolist())))    # 🐍 6주차 딕셔너리

💬 **관찰 질문**: 10개 클래스가 거의 고르게(5,400~6,700장) 있습니다. 3주차 용접 불량(10%)처럼 불균형하지 않으니 **정확도**를 주 지표로 써도 됩니다. 불균형했다면? — 혼동행렬·재현율을 봐야 합니다(6단계에서 혼동행렬은 어차피 봅니다).

## 3단계 — 훈련/검증 분할과 미니배치 `DataLoader`

6주차 규칙: **테스트는 봉인**, 에폭 수·하이퍼파라미터는 **검증 세트**로 고릅니다. 60,000장을 50,000(훈련) / 10,000(검증)으로 나누고, `DataLoader`로 128장씩 섞어서 꺼냅니다.

> **프롬프트 예시**
> "train_full을 random_split으로 50000/10000으로 나누고(generator 시드 0), 훈련은 batch_size=128 shuffle=True, 검증·테스트는 batch_size=1000 shuffle=False인 DataLoader 세 개를 만들어줘. 훈련 로더에서 배치 하나를 꺼내 이미지와 라벨의 shape, 한 에폭의 배치 수를 출력해줘.\"

In [ ]:
train_ds, val_ds = random_split(train_full, [50_000, 10_000], generator=torch.Generator().manual_seed(0))   # 🐍 키워드 인자

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)    # 훈련: 128장씩, 매 에폭 섞는다
val_loader   = DataLoader(val_ds,   batch_size=1000)                 # 검증·테스트: 섞을 필요 없음(shuffle 기본값 False), 크게 묶어 빠르게
test_loader  = DataLoader(test_ds,  batch_size=1000)

xb, yb = next(iter(train_loader))                                    # 📦 배치 하나 꺼내 보기: (이미지 묶음, 라벨 묶음)
print("배치 이미지:", tuple(xb.shape), "| 배치 라벨:", tuple(yb.shape), "| 라벨 앞 10개:", yb[:10].tolist())
print(f"한 에폭 = 훈련 {len(train_ds)}장 ÷ 128 = {len(train_loader)} 배치 → 한 에폭에 가중치를 {len(train_loader)}번 갱신")

💬 **관찰 질문**
1. 9주차에는 150점을 **한 번에** 넣어 한 에폭 = 한 걸음이었습니다. 오늘은 한 에폭 = **391걸음**. 왜 나눌까요? — 60,000장을 한 번에 넣으면 메모리가 모자라고, 작은 묶음으로 자주 걷는 쪽이 보통 더 빨리 수렴합니다(묶음마다 조금씩 다른 데이터라 "흔들리며 내려가는" 효과 — 확률적(Stochastic) 경사하강법의 S)
2. `shuffle=True`가 훈련에만 있는 이유 — 매 에폭 묶음 구성이 달라야 특정 순서를 외우지 않습니다. 검증·테스트는 채점만 하니 순서가 무관합니다

# Part B — 분류 모델 만들고 학습시키기 🧠

## 4단계 — 출력 10개: 점수(logit) → softmax → 예측

모델: `Flatten → Linear(784, 128) → ReLU → Linear(128, 10)`. 파라미터 784·128+128 + 128·10+10 = **101,770개**. 마지막 층 출력 10개는 "각 숫자일 점수"이고, 확률로 바꾸는 softmax는 **손실 부품 안에** 들어 있어 모델에는 넣지 않습니다.

> **프롬프트 예시**
> "nn.Sequential(Flatten, Linear(784,128), ReLU, Linear(128,10)) 모델을 만들어 파라미터 수를 출력하고, 학습 전 상태로 배치 xb를 넣은 출력 shape, 첫 장의 점수 10개, softmax 확률 10개(합 포함), argmax 예측과 실제 라벨을 출력해줘. nn.CrossEntropyLoss로 이 배치의 손실도 계산해줘.\"

In [ ]:
def make_model(hidden=128, dropout=0.0):
    layers = [nn.Flatten(), nn.Linear(784, hidden), nn.ReLU()]
    if dropout > 0: layers.append(nn.Dropout(dropout))      # 8단계에서 켠다. 0이면 없음
    layers.append(nn.Linear(hidden, 10))                     # 출력 10개 = 클래스 수. softmax는 여기 없다(손실 안에)
    return nn.Sequential(*layers).to(device)                 # 🐍 *layers: 리스트를 인자들로 풀어 넣기

torch.manual_seed(0)
model = make_model()
print(model); print("파라미터 수:", sum(p.numel() for p in model.parameters()))

logits = model(xb.to(device))                                # 📦 (128 × 10) — 배치 128장 × 클래스 10개의 '점수'
print("\n출력 shape:", tuple(logits.shape))
print("첫 장의 점수 10개   :", logits[0].detach().cpu().numpy().round(2))
probs = torch.softmax(logits, dim=1)                         # 📦 행마다(dim=1) 10개를 확률로 — 합이 1
print("softmax 확률 10개   :", probs[0].detach().cpu().numpy().round(3), "합 =", probs[0].sum().item())
print("예측 argmax:", probs[0].argmax().item(), "| 실제 라벨:", yb[0].item(), " ← 학습 전이니 맞으면 우연(1/10)")

loss_fn = nn.CrossEntropyLoss()                              # softmax + 로그 손실. 정답은 정수 라벨 그대로
print("학습 전 손실:", round(loss_fn(logits, yb.to(device)).item(), 3), " ← ln(10) ≈ 2.303 근처 = '열 개 중 아무거나' 수준")

💬 **관찰 질문**
1. 점수(logit)는 음수도 있고 합도 1이 아닙니다. softmax가 "큰 점수는 더 크게, 작은 점수는 0 근처로" 눌러 확률로 만듭니다. 3주차 시그모이드가 1개 점수를 0~1로 눌렀다면, softmax는 10개 점수를 **합 1**로 누릅니다
2. 학습 전 손실이 약 2.3인 이유: 확률이 10개에 고루 0.1씩이면 −ln(0.1) = 2.303. 학습이 되면 정답 칸 확률이 1에 가까워져 손실이 0으로 갑니다 — 5단계에서 이 숫자가 내려가는 것을 봅니다
3. `CrossEntropyLoss`에 라벨 `yb`(정수, 모양 (128,))를 그대로 넣었습니다. AI가 원-핫 인코딩을 하려 들면 "PyTorch CrossEntropyLoss는 정수 라벨을 받는다"고 알려 주세요

## 5단계 — 학습 루프에 "검증"을 끼우기: 에폭마다 두 손실 기록

9주차 루프 다섯 줄은 그대로이고, 바깥에 **배치 반복**이 한 겹, 에폭 끝에 **검증 세트 채점**이 한 번 추가됩니다. 평가 때는 `model.eval()` + `torch.no_grad()`(드롭아웃을 끄고, 기울기 기록도 끄기).

> **프롬프트 예시**
> "train_one_epoch(model, loader, loss_fn, opt)와 evaluate(model, loader, loss_fn) 함수를 만들어줘(evaluate는 평균 손실과 정확도 반환, model.eval()과 no_grad 사용). Adam(lr=1e-3)으로 5 에폭 학습하며 에폭마다 훈련 손실·검증 손실·검증 정확도를 출력하고 리스트에 기록해줘.\"

In [ ]:
def train_one_epoch(model, loader, loss_fn, opt):
    model.train()                                            # 학습 모드 (드롭아웃 켜짐)
    total, n = 0.0, 0
    for xb, yb in loader:                                    # 🐍 배치마다 (이미지, 라벨) 언패킹
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()                                      # ① 비우기
        loss = loss_fn(model(xb), yb)                        # ②③ 예측·채점
        loss.backward()                                      # ④ 미분
        opt.step()                                           # ⑤ 한 걸음   ← 9주차 다섯 줄 그대로
        total += loss.item() * len(xb); n += len(xb)
    return total / n                                         # 에폭 평균 훈련 손실

def evaluate(model, loader, loss_fn):
    model.eval()                                             # 평가 모드 (드롭아웃 꺼짐)
    total, correct, n = 0.0, 0, 0
    with torch.no_grad():                                    # 🐍 with: 블록 안에서만 기울기 기록 끔
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()   # 🐍 6주차 불리언 마스크 → True 개수
            n += len(xb)
    return total / n, correct / n                            # (평균 손실, 정확도)

def fit(model, train_loader, val_loader, epochs=5, lr=1e-3, verbose=True):
    "학습 전체: 에폭마다 훈련 손실·검증 손실·검증 정확도를 기록해 딕셔너리로 반환"
    loss_fn = nn.CrossEntropyLoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)   # 🐍 키워드 인자 lr
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}                               # 🐍 6주차 딕셔너리
    for epoch in range(1, epochs + 1):
        tl = train_one_epoch(model, train_loader, loss_fn, opt)
        vl, va = evaluate(model, val_loader, loss_fn)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl); hist["val_acc"].append(va)
        if verbose: print(f"epoch {epoch:2d} | train loss {tl:.4f} | val loss {vl:.4f} | val acc {va:.4f}")
    return hist

torch.manual_seed(0); model = make_model()
t0 = time.time(); hist = fit(model, train_loader, val_loader, epochs=5); print(f"학습 시간 {time.time()-t0:.0f}초 ({device})")

def plot_hist(hists, titles):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for h, t in zip(hists, titles):
        axes[0].plot(range(1, len(h["train_loss"]) + 1), h["train_loss"], "o-", label=f"{t} train"); axes[0].plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], "s--", label=f"{t} val")
        axes[1].plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], "o-", label=t)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Train vs validation loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val accuracy"); axes[1].set_title("Validation accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
plot_hist([hist], ["MLP-128"])

💬 **관찰 질문**
1. 손실이 2.3 → 0.1 안팎으로, 검증 정확도가 **97% 안팎**까지 올라갑니다(5주차 기계 진단 트리보다 높은 숫자지만 문제가 다릅니다). 단 5 에폭·CPU 수 분으로 — 이것이 9주차에 "열 줄"을 배운 이유
2. 훈련 손실과 검증 손실이 **아직 같이 내려가고** 있습니까? 그렇다면 과적합이 시작되지 않은 것입니다. 50,000장은 10만 개 파라미터가 외우기엔 많은 양이라 5 에폭으로는 과적합이 잘 안 보입니다 — 7단계에서 **일부러** 데이터를 줄여 과적합을 만듭니다
3. `model.train()`과 `model.eval()`은 지금 모델(드롭아웃 없음)에서는 차이가 없습니다. 8단계에서 의미가 생깁니다

## 6단계 — 🔒 테스트 한 번, 혼동행렬, 틀린 그림 보기

> **프롬프트 예시**
> "test_loader로 테스트 정확도를 한 번 출력하고, 예측과 정답으로 10×10 혼동행렬을 그려줘(sklearn confusion_matrix + ConfusionMatrixDisplay). 틀린 이미지 12장을 '예측/정답' 제목으로 격자에 보여줘.\"

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay      # 3주차 도구 재사용

test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss())
print(f"🔒 테스트 정확도 (한 번만): {test_acc:.4f}  (손실 {test_loss:.4f})")

model.eval(); preds, trues, wrong_imgs = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        p = model(xb.to(device)).argmax(dim=1).cpu()
        preds.append(p); trues.append(yb)
        m = p != yb                                                       # 🐍 불리언 마스크: 틀린 것만
        wrong_imgs += [(xb[i], p[i].item(), yb[i].item()) for i in torch.where(m)[0][:12 - len(wrong_imgs)]]
preds, trues = torch.cat(preds).numpy(), torch.cat(trues).numpy()

cm = confusion_matrix(trues, preds)
ConfusionMatrixDisplay(cm).plot(cmap="Blues", colorbar=False); plt.title("Confusion matrix — test set"); plt.show()
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape); print(f"가장 자주 헷갈린 쌍: 정답 {i} → 예측 {j} ({off[i, j]}건)")

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, (im, p, t) in zip(axes.ravel(), wrong_imgs[:12]):
    ax.imshow(im.squeeze(), cmap="gray"); ax.set_title(f"pred {p} / true {t}", fontsize=9, color="#e8590c"); ax.axis("off")
plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.show()

💬 **관찰 질문**
1. 혼동행렬의 대각선 밖에서 가장 큰 칸은 어디입니까? 보통 **4↔9, 3↔5, 7↔2**처럼 사람이 봐도 비슷한 모양입니다 — 3주차 "틀린 것은 경계에 있는 것"과 같은 이야기
2. 틀린 그림 중 "이건 사람도 헷갈리겠다" 싶은 것과 "이건 왜 틀렸지" 싶은 것을 나눠 보세요. 후자가 11주차 CNN이 개선하는 부분입니다(픽셀을 한 줄로 펴면서 **모양(위치 관계)** 정보가 사라졌기 때문)

# Part C — 과적합을 만들어 보고, 막아 보기 🛡️

## 7단계 — 과적합 재현: 데이터를 3,000장으로 줄이고 40 에폭

5단계에서 과적합이 안 보인 이유는 데이터가 많아서였습니다. 훈련 데이터를 **3,000장**으로 줄이고 은닉 뉴런을 256개로 늘려 40 에폭 돌리면, 모델이 3,000장을 **외우기** 시작합니다.

> **프롬프트 예시**
> "train_ds의 앞 3000장만 Subset으로 뽑아 batch_size=64 DataLoader를 만들고, make_model(hidden=256)을 40 에폭 학습해 훈련/검증 손실 곡선을 그려줘. 검증 손실이 최소인 에폭에 세로 점선과 'best' 주석을 달아줘.\"

In [ ]:
small_ds = Subset(train_ds, range(3000))                                  # 앞 3,000장만
small_loader = DataLoader(small_ds, batch_size=64, shuffle=True)

torch.manual_seed(0); model_over = make_model(hidden=256)
hist_over = fit(model_over, small_loader, val_loader, epochs=40, verbose=False)

def plot_overfit(h, title, ax=None):
    ax = ax or plt.subplots(figsize=(7, 4.2))[1]
    ep = range(1, len(h["val_loss"]) + 1); best = int(np.argmin(h["val_loss"])) + 1
    ax.plot(ep, h["train_loss"], "o-", ms=3, label="train loss"); ax.plot(ep, h["val_loss"], "s-", ms=3, label="val loss")
    ax.axvline(best, color="gray", ls="--"); ax.annotate(f"best val @ epoch {best}\n(val loss {h['val_loss'][best-1]:.3f})", xy=(best, h["val_loss"][best-1]), xytext=(best + 4, h["val_loss"][best-1] + 0.15), arrowprops=dict(arrowstyle="->"), fontsize=9)
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3); return best

best_over = plot_overfit(hist_over, "Overfitting: 3,000 images, hidden 256, 40 epochs"); plt.show()
print(f"최저 검증 손실 에폭: {best_over} (val loss {min(hist_over['val_loss']):.3f}, val acc {hist_over['val_acc'][best_over-1]:.3f})")
print(f"마지막(40) 에폭: train loss {hist_over['train_loss'][-1]:.4f} / val loss {hist_over['val_loss'][-1]:.3f} / val acc {hist_over['val_acc'][-1]:.3f}")

💬 **관찰 질문**
1. 훈련 손실은 **0 근처까지** 계속 내려가는데 검증 손실은 어느 에폭부터 **다시 올라갑니다**. 두 곡선이 벌어지는 그 지점이 과적합의 시작 — 5주차 깊이 무제한 트리(훈련 1.000 / 테스트 0.867), 6주차 KNN k=1과 **같은 현상**을 이번엔 에폭 축에서 봅니다
2. 검증 정확도는 생각보다 덜 떨어집니다(손실은 오르는데). 손실은 "얼마나 자신 있게 틀렸나"까지 재기 때문에 — 외운 모델은 **틀릴 때 더 확신을 가지고** 틀립니다. 그래서 과적합 감시는 정확도보다 **손실**로 합니다
3. "시험공부" 비유(6주차): 문제집 3,000문제를 40번 반복해 풀면 답을 외우지만 모의고사(검증)는 어느 순간부터 나빠집니다

## 8단계 — 드롭아웃: 학습 중 뉴런을 무작위로 끄기

`nn.Dropout(0.5)`은 **학습 때만** 은닉 뉴런의 절반을 매 배치 무작위로 0으로 만듭니다(`model.train()`). 평가 때(`model.eval()`)는 모두 켭니다. 어떤 뉴런이 꺼질지 모르니 특정 뉴런 몇 개에 **의존해 외우는 것이 불가능**해집니다.

> **프롬프트 예시**
> "같은 3000장·256뉴런 설정에 nn.Dropout(0.5)를 ReLU 뒤에 넣은 모델을 40 에폭 학습하고, 7단계 곡선과 나란히(1행 2열) 그려 비교해줘.\"

In [ ]:
torch.manual_seed(0); model_drop = make_model(hidden=256, dropout=0.5)
print(model_drop[3], "← ReLU 뒤에 끼운 드롭아웃 부품 (파라미터 0개)")
hist_drop = fit(model_drop, small_loader, val_loader, epochs=40, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
b1 = plot_overfit(hist_over, "No dropout", axes[0]); b2 = plot_overfit(hist_drop, "Dropout 0.5", axes[1]); plt.tight_layout(); plt.show()
print(pd.DataFrame({"": ["드롭아웃 없음", "드롭아웃 0.5"],
                    "최저 val loss": [min(hist_over["val_loss"]), min(hist_drop["val_loss"])],
                    "그 에폭": [b1, b2],
                    "40에폭 val loss": [hist_over["val_loss"][-1], hist_drop["val_loss"][-1]],
                    "40에폭 train loss": [hist_over["train_loss"][-1], hist_drop["train_loss"][-1]],
                    "40에폭 val acc": [hist_over["val_acc"][-1], hist_drop["val_acc"][-1]]}).round(4).to_string(index=False))

# 드롭아웃이 실제로 뉴런을 끄는지 눈으로: 같은 입력을 train 모드로 두 번 통과
model_drop.train(); h1 = model_drop[:4](xb[:1].to(device)); h2 = model_drop[:4](xb[:1].to(device))
print(f"\ntrain 모드: 은닉 256개 중 0인 뉴런 수 — 1회차 {(h1 == 0).sum().item()}개, 2회차 {(h2 == 0).sum().item()}개 (매번 다른 조합)")
model_drop.eval(); h3 = model_drop[:4](xb[:1].to(device)); print(f"eval 모드: 0인 뉴런 수 {(h3 == 0).sum().item()}개 (ReLU로 꺼진 것만, 드롭아웃은 꺼지지 않음)")

💬 **관찰 질문**
1. 드롭아웃을 넣으면 훈련 손실은 **덜** 내려가고(외우기가 방해받으니), 검증 손실은 **덜** 올라가거나 더 늦게 올라갑니다. 두 곡선의 **간격**이 좁아진 것이 핵심
2. 비유 — 팀 프로젝트에서 발표 직전까지 누가 빠질지 모르면 **한 사람에게 의존하는 계획을 세울 수 없고** 모두가 전체를 알아야 합니다. 드롭아웃은 뉴런에게 그 상황을 강제합니다. (과제 9 (a)의 비유를 여러분 말로 다시 쓰세요)
3. train/eval 모드 출력 비교: `model.eval()`을 잊고 평가하면 평가 때도 뉴런이 꺼져 정확도가 낮게 나옵니다 — AI 코드에서 `evaluate` 안에 `model.eval()`이 있는지 확인하세요

## 9단계 — 조기 종료: 검증 손실이 나빠지기 시작하면 멈추고, 가장 좋았던 가중치로 되돌리기

40 에폭을 다 돌릴 필요가 없었습니다. **검증 손실이 `patience`(예: 5) 에폭 연속 좋아지지 않으면 멈추고**, 가장 좋았던 시점의 `state_dict`를 복사해 두었다가 되돌립니다.

> **프롬프트 예시**
> "fit 함수에 조기 종료를 추가해줘: 검증 손실이 5 에폭 연속 개선되지 않으면 멈추고, 최저 검증 손실 시점의 state_dict(copy.deepcopy)를 보관했다가 마지막에 load_state_dict로 복원. 3000장·256뉴런 설정으로 돌려 몇 에폭에 멈췄는지와 복원 후 검증 정확도를 출력해줘.\"

In [ ]:
def fit_early_stop(model, train_loader, val_loader, max_epochs=40, lr=1e-3, patience=5):
    loss_fn = nn.CrossEntropyLoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, max_epochs + 1):
        tl = train_one_epoch(model, train_loader, loss_fn, opt); vl, va = evaluate(model, val_loader, loss_fn)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl); hist["val_acc"].append(va)
        if vl < best_loss:                                               # 좋아졌다 → 기록 갱신, 가중치 복사
            best_loss, best_state, best_epoch, wait = vl, copy.deepcopy(model.state_dict()), epoch, 0
        else:                                                            # 안 좋아졌다 → 기다린 횟수 +1
            wait += 1
            if wait >= patience:
                print(f"epoch {epoch}: 검증 손실이 {patience} 에폭 연속 개선 없음 → 조기 종료 (best = epoch {best_epoch})"); break
    model.load_state_dict(best_state)                                    # 📦 가장 좋았던 가중치로 되돌리기
    return hist, best_epoch

torch.manual_seed(0); model_es = make_model(hidden=256)
hist_es, best_ep = fit_early_stop(model_es, small_loader, val_loader)
vl, va = evaluate(model_es, val_loader, nn.CrossEntropyLoss())
print(f"복원 후 검증: loss {vl:.3f}, acc {va:.4f}  (7단계 40에폭 끝 모델의 val loss {hist_over['val_loss'][-1]:.3f})")
plot_overfit(hist_es, f"Early stopping (patience 5) — stopped at epoch {len(hist_es['val_loss'])}, restored epoch {best_ep}"); plt.show()

💬 **관찰 질문**
1. 조기 종료는 **에폭 수라는 하이퍼파라미터를 검증 세트로 자동으로 고르는 것**입니다 — 6주차 교차검증으로 k를 고른 것과 같은 논리. 테스트 세트로 멈출 시점을 정하면 반칙
2. `patience`를 1로 하면 너무 일찍(우연한 한 번의 상승에) 멈추고, 20으로 하면 조기 종료의 의미가 없습니다. 보통 5~10
3. `copy.deepcopy(model.state_dict())`가 필요한 이유: `state_dict()`는 **같은 텐서를 가리키는** 딕셔너리라 복사하지 않으면 이후 학습으로 값이 바뀝니다(9-2차시 3-1 "같은 객체" 콜백)

## 10단계 — 저장하고 불러오기: 현장에 배포하는 모양 💾

학습한 가중치(`state_dict`)를 파일로 저장하고, **새 모델 객체**에 불러와 같은 정확도가 나오는지 확인합니다. 학습 기록은 `json`으로 저장합니다 — 🐍 `with open(...)`.

> **프롬프트 예시**
> "model.state_dict()를 torch.save로 'mnist_mlp.pt'에 저장하고, hist 딕셔너리를 json 파일로 저장해줘. 그 다음 make_model()로 새 모델을 만들어 load_state_dict로 불러온 뒤 테스트 정확도가 원래 모델과 같은지 출력해줘. 파일 크기도 보여줘.\"

In [ ]:
import os
torch.save(model.state_dict(), "mnist_mlp.pt")                           # 가중치만 저장 (모델 구조는 코드에 있음)
with open("history.json", "w") as f:                                      # 🐍 with: 블록을 나오면 자동으로 닫힘
    json.dump(hist, f, indent=1)                                          # 딕셔너리 → 텍스트 파일
print(f"mnist_mlp.pt: {os.path.getsize('mnist_mlp.pt')/1e6:.2f} MB (파라미터 101,770개 × 4바이트) | history.json: {os.path.getsize('history.json')} bytes")

model_new = make_model()                                                  # 새 모델 — 무작위 가중치 상태
_, acc_before = evaluate(model_new, test_loader, nn.CrossEntropyLoss())
model_new.load_state_dict(torch.load("mnist_mlp.pt", map_location=device))   # 📦 파일 → 가중치 복원
_, acc_after = evaluate(model_new, test_loader, nn.CrossEntropyLoss())
print(f"새 모델 테스트 정확도: 불러오기 전 {acc_before:.4f} → 후 {acc_after:.4f}  (원래 모델 {test_acc:.4f}) → {'같다 ✅' if abs(acc_after - test_acc) < 1e-6 else '다르다'}")

with open("history.json") as f:                                           # 🐍 읽기 모드(기본값 "r")
    loaded = json.load(f)
print("불러온 기록의 키:", list(loaded.keys()), "| 마지막 val acc:", loaded["val_acc"][-1])

💬 **관찰 질문**
1. 저장한 것은 **가중치 숫자들**뿐입니다. 불러올 때 `make_model()`로 **같은 구조**를 먼저 만들어야 합니다 — 구조가 다르면(hidden=256) `load_state_dict`가 shape 불일치 에러를 냅니다(9-2차시 에러 표의 친척)
2. 현장 배포는 이 두 줄입니다: 학습용 컴퓨터에서 `torch.save`, 현장 장비에서 `load_state_dict` + `model.eval()` + `no_grad`로 예측만. 11주차에 Colab에서 학습한 CNN을 이 방식으로 저장해 봅니다

## 11단계 — 5·6·9·10주차 한눈에 비교 🗂️

| | 5주차 트리·숲 | 6주차 KNN·CV | 9주차 MLP 회귀 | **10주차 MLP 분류** |
|---|---|---|---|---|
| 데이터 | 기계 400대 × 센서 8 | 같음 | 응력-변형률 150점 | **이미지 60,000장 × 픽셀 784** |
| 출력 | 상태 4개 | 상태 4개 | 숫자 1개 | **점수 10개 → softmax** |
| 손실 | (불순도) | (거리) | MSE | **CrossEntropy** |
| 한 걸음의 데이터 | 전체 | 전체 | 전체(150) | **미니배치 128** |
| 복잡도 조절 | 깊이 | k | 뉴런 수 | **에폭 수(조기 종료)·드롭아웃·뉴런 수** |
| 과적합 신호 | 훈련 1.0 / 테스트↓ | k=1 훈련 1.0 | (안 봄 — 반칙) | **훈련 손실↓ 검증 손실↑** |
| 고르는 기준 | 테스트(반칙) → CV | CV | — | **검증 세트** |

## 12단계 — 자유 실험 🔬

- `make_model(hidden=32)`과 `hidden=512`로 5단계를 다시 — 정확도와 학습 시간은?
- `lr=1e-2`, `lr=1e-4`로 — 3주차 학습률 실험의 MNIST판 (너무 크면 손실이 요동, 너무 작으면 5 에폭에 못 올라옴)
- `batch_size=8`과 `2048`로 — 한 에폭의 걸음 수와 곡선의 매끄러움
- 7단계 과적합 실험에서 3,000장을 1,000장 / 10,000장으로 — 과적합이 시작되는 에폭이 어떻게 움직이나
- 은닉층을 하나 더(`Linear(128,128), ReLU`) 끼우기 — 정확도가 오르나? 파라미터 수는?
- (보너스) 종이에 숫자를 써서 사진 찍어 올리고 28×28 흑백으로 바꿔 예측해 보기 — AI에게 "PIL로 이미지를 28×28 grayscale 텐서로 변환하는 코드" 요청. 배경이 흰색이면 MNIST(검정 배경)와 반대라 반전이 필요합니다

In [ ]:
# 자유 실험: AI가 만들어준 코드를 붙여넣고 실행하세요

## 마무리

1. **분류 = 출력 클래스 수만큼**, softmax가 점수를 확률로, `CrossEntropyLoss`가 그것을 한 부품에. 정답은 정수 라벨 그대로
2. 이미지는 **784개 센서**다 — `Flatten`으로 펴면 지금까지의 모든 도구가 그대로 쓰인다(모양 정보는 잃는다 → 11주차 CNN)
3. **미니배치·에폭**: `DataLoader`가 섞어서 나눠 주고, 9주차 다섯 줄 루프가 배치마다 돈다
4. **학습 곡선 두 개**(훈련·검증 손실)가 벌어지는 지점 = 과적합. **조기 종료**는 그 지점에서 멈추는 것, **드롭아웃**은 외우기를 방해해 간격을 좁히는 것. 둘 다 **검증 세트**로 판단한다 — 테스트는 마지막 한 번
5. `state_dict`를 **저장·불러오기**하면 현장에 배포할 수 있다 — 구조(코드) + 가중치(파일)

**과제 9 "과적합 관찰 일지"**는 `week10_hw.ipynb`에서 — 오늘 7·8단계(3,000장·40에폭, 드롭아웃 없음/있음)를 그대로 돌려 두 곡선을 그리고, 과적합이 시작되는 지점을 **화살표 주석으로 표시**한 뒤, 드롭아웃이 과적합을 막는 원리를 팀 프로젝트 비유로 쓰면 됩니다. `make_model`·`fit`·`plot_overfit` 함수는 복사해 가세요.